# Ejercicio Guiado: Accesibilidad a Servicios de Emergencia

**IIC2115 — Programación como Herramienta para la Ingeniería**  
**Capítulo 3:Clase integradora**

En este ejercicio recorreremos un problema completo que integra todo lo visto en el capítulo: carga y procesamiento de datos geoespaciales con GeoPandas, descarga y análisis de redes viales con OSMnx/NetworkX, y visualización.

**Problema:** Una autoridad municipal quiere evaluar la cobertura de servicios de emergencia (bomberos, ambulancias) en la comuna de **San Joaquín**. Nos pide:
1. Identificar las zonas con mayor concentración de accidentes.
2. Evaluar qué tan bien cubiertas están esas zonas desde puntos de respuesta candidatos.
3. Recomendar la mejor ubicación para un nuevo centro de emergencias.


## Paso 0: Importaciones y descarga de datos

In [ ]:
!pip install osmnx
!pip install geopandas
!pip install networkx

In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import urllib.request
import zipfile
import os
from shapely.geometry import Point

In [ ]:
# descargar datos del curso desde github (solo si no existen)
if not os.path.exists("datos_geoespaciales"):
    zip_url = "https://raw.githubusercontent.com/IIC2115/Syllabus/main/Material%20de%20clases/Cap%C3%ADtulo%203/Notebooks/datos_geoespaciales.zip"
    urllib.request.urlretrieve(zip_url, "datos_geoespaciales.zip")
    with zipfile.ZipFile("datos_geoespaciales.zip", "r") as z:
        z.extractall(".")
    print("datos descargados y extraídos")
else:
    print("datos ya disponibles")


## Paso 1: Carga de datos geoespaciales

In [ ]:
# cargar datos base
comunas = gpd.read_file("datos_geoespaciales/Comunas/comunas.shp")
distritos = gpd.read_file("datos_geoespaciales/Distritos Censales/Distritos Censales RM.shp")
accidentes = gpd.read_file("datos_geoespaciales/Accidentes/Puntos_criticos_2019.geojson")

print(f"comunas: {len(comunas)}")
print(f"distritos: {len(distritos)}")
print(f"accidentes: {len(accidentes)}")

### 1.1 Filtrar el área de estudio

Filtramos todo para la comuna de San Joaquín. Esto ya lo saben hacer de los E1/E2 con GeoPandas.

In [ ]:
# Asegurar la carga y alineación de sistemas de coordenadas (CRS)
comunas_4326 = comunas.to_crs("EPSG:4326")
distritos_4326 = distritos.to_crs("EPSG:4326")
accidentes_4326 = accidentes.to_crs("EPSG:4326")

# filtrar comuna de san joaquín
sj = comunas_4326[comunas_4326["Comuna"] == "San Joaquín"].copy()

# filtrar distritos dentro de la comuna
distritos_sj = gpd.sjoin(distritos_4326, sj, predicate="intersects")

# filtrar accidentes dentro de la comuna
accidentes_sj = gpd.sjoin(accidentes_4326, sj, predicate="intersects")

print(f"distritos en san joaquín: {len(distritos_sj)}")
print(f"accidentes en san joaquín: {len(accidentes_sj)}")


## Paso 2: Análisis espacial — zonas críticas de accidentes

Identificamos qué distritos concentran más accidentes. (Uso de Geopandas)

In [ ]:
# Limpiar columnas de uniones previas si existen para evitar conflictos
if 'index_right' in accidentes_sj.columns:
    accidentes_sj = accidentes_sj.drop(columns=['index_right'])
if 'index_right' in distritos_sj.columns:
    distritos_sj = distritos_sj.drop(columns=['index_right'])
if 'index_left' in distritos_sj.columns:
    distritos_sj = distritos_sj.drop(columns=['index_left'])

# contar accidentes por distrito
accidentes_por_distrito = gpd.sjoin(
    accidentes_sj, distritos_sj, predicate="within"
).groupby("index_right").size().reset_index(name="n_accidentes")

# unir al geodataframe de distritos de forma segura
distritos_sj = distritos_sj.merge(
    accidentes_por_distrito, left_index=True, right_on="index_right", how="left"
)
distritos_sj["n_accidentes"] = distritos_sj["n_accidentes"].fillna(0)

# visualizar
fig, ax = plt.subplots(figsize=(8, 8))
distritos_sj.plot(column="n_accidentes", cmap="YlOrRd", legend=True,
                  edgecolor="gray", linewidth=0.3, ax=ax)
ax.set_title("accidentes por distrito censal — san joaquín")
ax.set_axis_off()
plt.tight_layout()
plt.show()


## Paso 3: Descarga de la red vial

Ahora pasamos a OSMnx. Descargamos la red vehicular de la comuna.

In [ ]:
# descargar red vehicular de san joaquín
grafo = ox.graph_from_place("San Joaquín, Santiago, Chile", network_type="drive")

print(f"nodos: {grafo.number_of_nodes()}")
print(f"aristas: {grafo.number_of_edges()}")

In [ ]:
# visualización rápida
fig, ax = ox.plot_graph(grafo, node_size=0, edge_color="gray",
                        edge_linewidth=0.5, figsize=(8, 8))


## Paso 4: Definir puntos candidatos para centro de emergencias

Proponemos 3 ubicaciones candidatas. En un caso real, esto vendría de un análisis más detallado, pero aquí elegimos puntos representativos distribuidos en la comuna.

In [ ]:
# definir 3 puntos candidatos (lat, lon)
# ajustar coordenadas según la comuna real
candidatos = {
    "candidato_norte": (-33.480, -70.625),
    "candidato_centro": (-33.490, -70.630),
    "candidato_sur": (-33.500, -70.635),
}

# encontrar nodos más cercanos en la red
nodos_candidatos = {}
for nombre, (lat, lon) in candidatos.items():
    nodo = ox.nearest_nodes(grafo, X=lon, Y=lat)
    nodos_candidatos[nombre] = nodo
    print(f"{nombre}: nodo {nodo}")


## Paso 5: Calcular tiempos de respuesta

Agregamos velocidades y tiempos de viaje estimados a la red, y calculamos desde cada candidato el tiempo de llegada a las zonas con accidentes.

In [ ]:
# agregar velocidades y tiempos de viaje (compatible con OSMnx v2.x y v1.x)
try:
    # Para OSMnx v2.x
    grafo = ox.add_edge_speeds(grafo)
    grafo = ox.add_edge_travel_times(grafo)
except AttributeError:
    # Para OSMnx v1.x
    grafo = ox.routing.speed.add_edge_speeds(grafo)
    grafo = ox.routing.speed.add_edge_travel_times(grafo)

In [ ]:
# obtener los nodos más cercanos a cada accidente
nodos_accidentes = [ox.nearest_nodes(grafo, X=p.x, Y=p.y)
                    for p in accidentes_sj.geometry]

# tomar nodos únicos para eficiencia
nodos_accidentes_unicos = list(set(nodos_accidentes))
print(f"nodos únicos de accidentes: {len(nodos_accidentes_unicos)}")

In [ ]:
# calcular tiempo de viaje desde cada candidato a cada nodo de accidente
resultados = {}

for nombre, nodo_origen in nodos_candidatos.items():
    tiempos = []
    for nodo_destino in nodos_accidentes_unicos:
        try:
            t = nx.shortest_path_length(grafo, nodo_origen, nodo_destino,
                                         weight="travel_time")
            tiempos.append(t)
        except nx.NetworkXNoPath:
            tiempos.append(np.inf)

    tiempos_validos = [t for t in tiempos if t < np.inf]
    resultados[nombre] = {
        "tiempo_promedio_seg": np.mean(tiempos_validos),
        "tiempo_maximo_seg": np.max(tiempos_validos),
        "cobertura_5min_pct": sum(1 for t in tiempos_validos if t <= 300) / len(tiempos_validos) * 100,
    }

# mostrar resultados
df_resultados = pd.DataFrame(resultados).T
df_resultados["tiempo_promedio"] = df_resultados["tiempo_promedio_seg"].apply(
    lambda x: f"{x:.0f} seg ({x/60:.1f} min)")
df_resultados["tiempo_maximo"] = df_resultados["tiempo_maximo_seg"].apply(
    lambda x: f"{x:.0f} seg ({x/60:.1f} min)")
df_resultados["cobertura_5min"] = df_resultados["cobertura_5min_pct"].apply(
    lambda x: f"{x:.1f}%")
print(df_resultados[["tiempo_promedio", "tiempo_maximo", "cobertura_5min"]])


## Paso 6: Análisis de centralidad del candidato

Evaluamos la posición estructural de cada candidato en la red usando cercanía y excentricidad.

In [ ]:
# centralidad de cercanía para los nodos candidatos
closeness = nx.closeness_centrality(grafo, distance="length")

for nombre, nodo in nodos_candidatos.items():
    print(f"{nombre}: closeness = {closeness[nodo]:.6f}")

In [ ]:
# excentricidad: trabajar con el componente conexo más grande si es necesario
if nx.is_strongly_connected(grafo):
    subgrafo = grafo
else:
    print("el grafo no es fuertemente conexo, usando componente más grande...")
    nodos_componente = max(nx.strongly_connected_components(grafo), key=len)
    subgrafo = grafo.subgraph(nodos_componente).copy()

for nombre, nodo in nodos_candidatos.items():
    if nodo in subgrafo:
        ecc = nx.eccentricity(subgrafo, v=nodo, weight="length")
        print(f"{nombre}: excentricidad = {ecc:.0f} m ({ecc/1000:.1f} km)")
    else:
        print(f"{nombre}: nodo fuera del componente conexo principal")


## Paso 7: Visualización integradora

Combinamos todo en un mapa final: distritos coloreados por accidentalidad, la red vial, los candidatos, y la ruta desde el mejor candidato al accidente más lejano.

In [ ]:
# elegir el mejor candidato (menor tiempo promedio)
mejor = min(resultados.keys(),
            key=lambda k: resultados[k]["tiempo_promedio_seg"])
print(f"mejor candidato: {mejor}")

# encontrar el nodo de accidente más lejano desde el mejor candidato
nodo_mejor = nodos_candidatos[mejor]
tiempos_mejor = []
for nd in nodos_accidentes_unicos:
    try:
        t = nx.shortest_path_length(grafo, nodo_mejor, nd, weight="travel_time")
        tiempos_mejor.append((nd, t))
    except nx.NetworkXNoPath:
        pass

nodo_lejano = max(tiempos_mejor, key=lambda x: x[1])[0]
ruta_critica = nx.shortest_path(grafo, nodo_mejor, nodo_lejano, weight="travel_time")

In [ ]:
# mapa final integrador
fig, ax = plt.subplots(figsize=(12, 12))

# capa 1: distritos por accidentalidad
distritos_sj.plot(column="n_accidentes", cmap="YlOrRd", alpha=0.5,
                  edgecolor="gray", linewidth=0.3, ax=ax, legend=True,
                  legend_kwds={"label": "accidentes por distrito"})

# capa 2: red vial
aristas_gdf = ox.graph_to_gdfs(grafo, nodes=False)
aristas_gdf.plot(ax=ax, color="lightgray", linewidth=0.3)

# capa 3: ruta crítica
ruta_edges = list(zip(ruta_critica[:-1], ruta_critica[1:]))
ruta_gdf = aristas_gdf.loc[aristas_gdf.index.isin(
    [(u, v, 0) for u, v in ruta_edges])]
ruta_gdf.plot(ax=ax, color="blue", linewidth=2.5, label="ruta crítica")

# capa 4: candidatos
colores_candidatos = {"candidato_norte": "green", "candidato_centro": "orange",
                      "candidato_sur": "purple"}
nodos_gdf = ox.graph_to_gdfs(grafo, edges=False)
for nombre, nodo in nodos_candidatos.items():
    punto = nodos_gdf.loc[nodo]
    ax.plot(punto.geometry.x, punto.geometry.y, "o", color=colores_candidatos[nombre],
            markersize=12, label=nombre, zorder=5)

# resaltar el mejor
punto_mejor = nodos_gdf.loc[nodo_mejor]
ax.plot(punto_mejor.geometry.x, punto_mejor.geometry.y, "*",
        color="red", markersize=20, label=f"★ {mejor} (recomendado)", zorder=6)

ax.legend(loc="lower left", fontsize=9)
ax.set_title("evaluación de accesibilidad a emergencias — san joaquín", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.show()


## ¿Qué hicimos?

En este ejercicio integramos:

- **GeoPandas**: carga de datos, filtrado espacial, spatial joins, conteo por distrito, visualización de capas.
- **OSMnx**: descarga de red vial, nodos más cercanos, velocidades y tiempos de viaje.
- **NetworkX**: rutas más cortas, tiempos de viaje, centralidad de cercanía, excentricidad.

Este es el tipo de análisis integrado que se espera en el laboratorio.